# 🧬 Bayesian Universal Differential Equations (UDEs): Embracing the Unknown
To fully understand **Bayesian Universal Differential Equations**, we must look at how they blend deep learning with known physics, and why *probability* is far superior to standard *optimization* when dealing with the unknown.

---

## 1. The Foundation: What is a UDE?
A standard Universal Differential Equation (UDE) is a hybrid model. It combines the physical or biological mechanisms we *already understand* with a neural network that acts as a universal approximator for the *physics we don't understand*.

Mathematically, it is expressed as:
$$ \frac{du}{dt} = f_{known}(u, t, p) + NN_\theta(u, t) $$

*   **$f_{known}$**: The known physics (e.g., standard protein decay, basic diffusion).
*   **$NN_\theta$**: A neural network (parameterized by weights $\theta$) acting as a stand-in for the missing dynamics (e.g., an unknown growth mechanism).

### The Deterministic Flaw
Standard UDEs use gradient descent to find a *single* set of optimal weights ($\theta^*$) that perfectly fits the data. However, real-world data is noisy, sparse, or incomplete. A deterministic UDE will give you one highly confident answer, but it cannot tell you if that answer is a robust scientific discovery or if the neural network is simply hallucinating to fit the noise.

## 2. The Bayesian Shift: Probability Over Optimization
A Bayesian UDE abandons the search for a single "best" answer. Instead of fixed numbers, it treats the neural network's weights ($\theta$) and any unknown physical parameters ($p$) as **probability distributions**. 

Using **Bayes' Theorem**, the goal is to compute the **Posterior Distribution**:
$$ P(\theta, p \mid \mathcal{D}) \propto P(\mathcal{D} \mid \theta, p) \cdot P(\theta, p) $$

This breaks down into three critical components:

1.  **The Prior ($P(\theta, p)$):** This encodes our scientific beliefs *before* seeing data. We might enforce that a biological decay rate must be strictly positive, or apply a standard normal distribution to the neural network weights. This acts as Occam's Razor, pushing the network toward simpler functions unless the data demands complexity.
2.  **The Likelihood ($P(\mathcal{D} \mid \theta, p)$):** This asks the forward question: *If we assume these specific weights, how likely is the ODE to produce the data we actually measured?* This accounts for the inherent measurement noise ($\sigma^2$) in our instruments.
3.  **The Posterior ($P(\theta, p \mid \mathcal{D})$):** The final output. Because the neural network's weights are now a distribution, its output is no longer a single curve, but an **ensemble of curves**.

## 3. Computational Inference: How We Solve It
Because differential equations are highly non-linear, we cannot solve this analytically. We must use advanced computational sampling techniques:

*   **Hamiltonian Monte Carlo (HMC) & NUTS:** The No-U-Turn Sampler (NUTS) calculates gradients through the ODE solver (using adjoint sensitivity methods) to intelligently explore the high-dimensional space of the neural network. It is mathematically rigorous and explores the true posterior.
*   **Variational Inference (VI):** For massive networks where MCMC is too slow, VI approximates the complex posterior by finding a simpler, known distribution (like a Gaussian) that closely matches it by minimizing the Kullback-Leibler (KL) divergence.

## 4. The Output: Uncertainty Quantification
When you run a Bayesian UDE, you do not plot one line. You plot hundreds of valid trajectories. This allows you to separate and visualize two distinct types of uncertainty:

*   **Aleatoric Uncertainty:** The unavoidable, inherent noise in your data measurements (the jitter in your sensors).
*   **Epistemic Uncertainty:** The model's ignorance. The variance in the neural network's predictions. 

**The Visual Proof:** Where you have plenty of data, the ensemble of curves will be tightly packed together. But where data is missing (e.g., forecasting into the future), the curves will **fan out wildly**. This visually and mathematically proves what the model *does not know*.

## 5. The Ultimate Value: The SciML Pipeline
In Scientific Machine Learning (SciML), training the UDE is only Step 1. Step 2 is using **Symbolic Regression** (like SINDy) to turn the neural network back into a human-readable mathematical equation.

*   If you pass a **Deterministic UDE** to SINDy, it blindly fits an equation to a single curve—even if that curve was just overfitting noise in a data-sparse region.
*   If you pass a **Bayesian UDE** to SINDy, it receives an ensemble. If the curves diverge wildly in a certain region, SINDy knows to assign lower confidence to its regression there. 

The final result is a rigorously discovered physical equation with publication-ready confidence intervals! 
*(e.g., $\frac{du}{dt} = -0.5u + (4.2 \pm 0.3)u^2$)*

In [ ]:
using DifferentialEquations, SciMLSensitivity, Lux, ComponentArrays, Random
using Turing, Distributions, LinearAlgebra

# 1. The True Physics (Ground Truth)
function true_dynamics!(du, u, p, t)
    # -0.5*u is known decay. 0.1*u^2 is the UNKNOWN mechanism.
    du[1] = -0.5 * u[1] + 0.1 * u[1]^2 
end

u0 = [1.0]                     # Initial protein concentration
tspan = (0.0, 10.0)            # 10 seconds of observation
prob_true = ODEProblem(true_dynamics!, u0, tspan)
sol_true = solve(prob_true, Tsit5(), saveat=0.5)
t_steps = sol_true.t

# Add 10% biological measurement noise
data = Array(sol_true) + 0.1 * randn(size(Array(sol_true)))

In [ ]:
# 2. Define the Neural Network (The Universal Approximator)
rng = Random.default_rng()
# A tiny MLP: 1 input -> 10 hidden nodes -> 1 output
U_net = Lux.Chain(Lux.Dense(1, 10, tanh), Lux.Dense(10, 1))
p_nn, st = Lux.setup(rng, U_net)

# Flatten the neural network weights into a 1D array for the ODE solver
θ_initial = ComponentArray(p_nn) 

# 3. Define the UDE
function ude_dynamics!(du, u, θ, t)
    known_term = -0.5 * u[1]
    
    # The NN predicts the missing physics based on the current state (u)
    # θ represents the weights we need to sample
    unknown_term = U_net([u[1]], θ, st)[1][1] 
    
    du[1] = known_term + unknown_term
end

prob_ude = ODEProblem(ude_dynamics!, u0, tspan, θ_initial)

In [ ]:
@model function bayesian_ude(data, prob, t_steps)
    # A. THE PRIORS
    # How many weights are in the Neural Network?
    p_len = length(prob.p)
    
    # 1. Prior for the Neural Network Weights (θ)
    # We assume weights are normally distributed around 0 (Regularization)
    θ ~ MvNormal(zeros(p_len), 1.0 * I)
    
    # 2. Prior for the measurement noise (σ)
    # We expect some noise, but strictly positive
    σ ~ InverseGamma(2, 3) 

    # Update the ODE problem with the currently sampled weights
    new_prob = remake(prob, p=θ)

    # B. THE FORWARD PASS (ODE Solve)
    # We use an Adjoint method so Turing can calculate gradients through the ODE
    predicted = solve(new_prob, Tsit5(), saveat=t_steps, 
                      sensealg=InterpolatingAdjoint(autojacvec=ReverseDiffVJP(true)))

    # If the sampled weights cause the ODE to blow up to infinity, reject the sample
    if predicted.retcode != ReturnCode.Success
        Turing.@addlogprob! -Inf
        return
    end

    # C. THE LIKELIHOOD 
    # Compare the ODE prediction against the noisy data
    for i in 1:length(t_steps)
        # "The probability of observing this data point is a Normal distribution 
        # centered on the ODE's prediction, with variance σ"
        data[1, i] ~ Normal(predicted[1, i], σ)
    end
end

# Instantiate the model
probabilistic_model = bayesian_ude(data, prob_ude, t_steps)

In [ ]:
# Run the NUTS Sampler
# Note: Because it calculates gradients through an ODE solver at every step, 
# this requires heavy computation!
chain = sample(probabilistic_model, NUTS(0.65), 500)